# 01 - Data Exploration and Preprocessing
This notebook explores COVID-19 data, performs preprocessing, missing value handling, feature engineering, and sanity checks.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.covid_analyzer import COVIDAnalyzer
plt.style.use('seaborn-v0_8')
sns.set_context('talk')

In [ ]:
# Load sample data using the project analyzer (generates realistic synthetic data)
analyzer = COVIDAnalyzer()
covid_df = analyzer.load_data(generate_sample=True)
covid_df.head()

## Basic Profiling

In [ ]:
covid_df.info()
covid_df.describe().T.head(20)

## Missing Values and Duplicates

In [ ]:
covid_df.isna().mean().sort_values(ascending=False)
covid_df.duplicated().sum()

## Feature Engineering
- 7-day rolling metrics
- Case fatality ratio (CFR)
- Growth rates

In [ ]:
covid_df = covid_df.sort_values(['country','date'])
covid_df['cases_7d'] = covid_df.groupby('country')['daily_cases'].transform(lambda s: s.rolling(7).mean())
covid_df['deaths_7d'] = covid_df.groupby('country')['daily_deaths'].transform(lambda s: s.rolling(7).mean())
covid_df['cfr'] = np.where(covid_df['total_cases']>0, covid_df['total_deaths']/covid_df['total_cases'], np.nan)
covid_df['case_growth'] = covid_df.groupby('country')['total_cases'].pct_change().replace([np.inf,-np.inf], np.nan)
covid_df.head()

## Visual EDA

In [ ]:
top_countries = (covid_df.groupby('country')['total_cases']
                 .max().sort_values(ascending=False).head(10).index)
fig, ax = plt.subplots(1,2, figsize=(16,6))
sns.barplot(x='total_cases', y='country', data=(covid_df[covid_df['country'].isin(top_countries)]
             .groupby('country', as_index=False)['total_cases'].max()
            ), ax=ax[0])
ax[0].set_title('Top 10 Countries by Total Cases')
sample_c = np.random.choice(top_countries, 1)[0]
tmp = covid_df[covid_df['country']==sample_c]
ax[1].plot(tmp['date'], tmp['cases_7d'], label='Cases 7d')
ax[1].plot(tmp['date'], tmp['deaths_7d'], label='Deaths 7d')
ax[1].legend(); ax[1].set_title(f'Trend - {sample_c}'); plt.tight_layout()

## Save Cleaned Snapshot

In [ ]:
import os
os.makedirs('results', exist_ok=True)
covid_df.to_csv('results/cleaned_covid_sample.csv', index=False)
covid_df.head()